# OptiCell Stage 2 — CTC time-lapse (Colab)

Runs **QC → segment → features → phenotype → tracking** on Cell Tracking Challenge 2D training sequences, **one dataset at a time**.

| Dataset | Size | Notes |
|---------|------|-------|
| Fluo-N2DH-GOWT1 | ~53 MB | GFP nuclei, mouse stem cells |
| Fluo-N2DH-SIM+ | ~91 MB | Simulated Hoechst nuclei + GT |
| Fluo-N2DL-HeLa | ~182 MB | HeLa H2b-GFP |

**Policy:** measured outputs only. Cite CTC (Nature Methods) if you publish.

Runtime: **CPU is fine** for threshold backend. GPU optional (not required).

## 0. Setup

In [ ]:
import os, sys, zipfile, urllib.request, json, shutil
from pathlib import Path

# Optional: keep outputs on Drive across sessions
USE_DRIVE = False  # set True to mount Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/opticell_stage2')
else:
    BASE = Path('/content/opticell_stage2')

BASE.mkdir(parents=True, exist_ok=True)
os.chdir('/content')
print('BASE =', BASE)

In [ ]:
# Clone + install OptiCell (threshold backend only — no Cellpose needed)
REPO = Path('/content/Virelion-OptiCell')
if not REPO.exists():
    !git clone https://github.com/Virelion-Biotech/Virelion-OptiCell.git
else:
    !git -C /content/Virelion-OptiCell pull origin main

%cd /content/Virelion-OptiCell
!pip install -e . -q
print('OptiCell ready')

## 1. Dataset catalog + download helper

Training zips from https://data.celltrackingchallenge.net/training-datasets/

In [ ]:
DATASETS = {
    'Fluo-N2DH-GOWT1': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-GOWT1.zip',
        'mb': 53,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DH-SIM+': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DH-SIM+.zip',
        'mb': 91,
        'sequences': ['01', '02'],
    },
    'Fluo-N2DL-HeLa': {
        'url': 'https://data.celltrackingchallenge.net/training-datasets/Fluo-N2DL-HeLa.zip',
        'mb': 182,
        'sequences': ['01', '02'],
    },
}

CTC_ROOT = BASE / 'ctc'
CTC_ROOT.mkdir(parents=True, exist_ok=True)

def download_and_extract(name: str) -> Path:
    meta = DATASETS[name]
    dest_dir = CTC_ROOT / name
    zip_path = CTC_ROOT / f'{name}.zip'
    if dest_dir.exists() and any(dest_dir.iterdir()):
        print(f'[skip download] {name} already at {dest_dir}')
        return dest_dir
    print(f'[download] {name} (~{meta["mb"]} MB) ...')
    urllib.request.urlretrieve(meta['url'], zip_path)
    print(f'[unzip] {zip_path}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(CTC_ROOT)
    # CTC zips usually extract to CTC_ROOT/name
    if not dest_dir.exists():
        # fallback: find folder
        cands = [p for p in CTC_ROOT.iterdir() if p.is_dir() and name.split('+')[0] in p.name]
        if cands:
            print('found', cands[0])
            return cands[0]
        raise FileNotFoundError(f'Extracted folder missing for {name}')
    print(f'[ok] {dest_dir}')
    return dest_dir

print('Datasets:', list(DATASETS))

## 2. Run Stage-2 on one sequence

CTC frames live in `01/`, `02/`, ... as `t000.tif`, `t001.tif`, ...

In [ ]:
import subprocess

def run_stage2(dataset_name: str, sequence: str, max_frames: int = 0):
    """Download dataset if needed, run killer workflow with tracking."""
    root = download_and_extract(dataset_name)
    seq_dir = root / sequence
    if not seq_dir.is_dir():
        # some extracts nest one more level
        alt = root / dataset_name / sequence
        seq_dir = alt if alt.is_dir() else seq_dir
    if not seq_dir.is_dir():
        print('Available under', root, ':', list(root.iterdir())[:20])
        raise FileNotFoundError(seq_dir)

    frames = sorted(seq_dir.glob('*.tif')) + sorted(seq_dir.glob('*.tiff'))
    frames = [f for f in frames if not f.name.startswith('._')]
    print(f'{dataset_name}/{sequence}: {len(frames)} frames in {seq_dir}')
    if not frames:
        raise RuntimeError('no frames')

    out = BASE / 'outputs' / f'stage2_{dataset_name}_{sequence}'
    out.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        '/content/Virelion-OptiCell/scripts/run_killer_workflow.py',
        str(seq_dir),
        '-o', str(out),
        '--backend', 'threshold',
        '--enable-tracking',
        '--track-max-distance', '30',
        '--track-max-gap', '1',
    ]
    if max_frames and max_frames > 0:
        cmd += ['--max-images', str(max_frames)]

    print('CMD:', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-4000:] if r.stdout else '')
    if r.returncode != 0:
        print(r.stderr[-2000:] if r.stderr else '')
        raise RuntimeError(f'Stage2 failed code={r.returncode}')

    summary_path = out / 'workflow_summary.json'
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print('=== SUMMARY ===')
        for k, v in summary.get('summary', {}).items():
            print(f'  {k}: {v}')
        return summary
    return None

print('run_stage2() defined')

## 3. Run datasets **one by one**

Execute cells below in order. Start with GOWT1 (smallest).

Tip: for a quick smoke test set `MAX_FRAMES = 20`; for full sequence use `0`.

In [ ]:
MAX_FRAMES = 0  # 0 = all frames in the sequence

# --- 1/3 GOWT1 sequence 01 ---
s1 = run_stage2('Fluo-N2DH-GOWT1', '01', max_frames=MAX_FRAMES)

In [ ]:
# --- 2/3 GOWT1 sequence 02 ---
s2 = run_stage2('Fluo-N2DH-GOWT1', '02', max_frames=MAX_FRAMES)

In [ ]:
# --- 3/6 SIM+ sequence 01 ---
s3 = run_stage2('Fluo-N2DH-SIM+', '01', max_frames=MAX_FRAMES)

In [ ]:
# --- 4/6 SIM+ sequence 02 ---
s4 = run_stage2('Fluo-N2DH-SIM+', '02', max_frames=MAX_FRAMES)

In [ ]:
# --- 5/6 HeLa sequence 01 (larger download) ---
s5 = run_stage2('Fluo-N2DL-HeLa', '01', max_frames=MAX_FRAMES)

In [ ]:
# --- 6/6 HeLa sequence 02 ---
s6 = run_stage2('Fluo-N2DL-HeLa', '02', max_frames=MAX_FRAMES)

## 4. Aggregate measured summaries only

In [ ]:
import pandas as pd

rows = []
out_root = BASE / 'outputs'
for p in sorted(out_root.glob('stage2_*/workflow_summary.json')):
    data = json.loads(p.read_text())
    s = data.get('summary', {})
    rows.append({
        'run': p.parent.name,
        'backend': data.get('backend'),
        'n_images': s.get('n_images'),
        'mean_object_count': s.get('mean_object_count'),
        'mean_confidence': s.get('mean_confidence'),
        'mean_focus': s.get('mean_focus'),
        'n_objects_total': s.get('n_objects_total'),
        'n_tracks': s.get('n_tracks'),
        'tracking_enabled': s.get('tracking_enabled'),
        'phenotype_positive_fraction': s.get('phenotype_positive_fraction'),
    })

table = pd.DataFrame(rows)
display(table)
agg_path = out_root / 'stage2_ctc_aggregate.csv'
table.to_csv(agg_path, index=False)
print('Wrote', agg_path)
print('Copy these numbers only — no invented TRA scores unless you add GT scoring.')

## 5. Download results (optional)

Zip one run or everything under `BASE/outputs`.

In [ ]:
from google.colab import files

zip_out = '/content/stage2_ctc_outputs.zip'
!cd {BASE} && zip -r -q {zip_out} outputs
print('Created', zip_out)
# files.download(zip_out)  # uncomment to download in browser

### Cite

Cell Tracking Challenge data — see https://celltrackingchallenge.net/datasets/ and the Nature Methods CTC papers.

OptiCell: https://github.com/Virelion-Biotech/Virelion-OptiCell